[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/t05_definite.ipynb)

# The integral that takes three lines: definite integrals and area

∫₀^(π/2) sin x ÷ (sin x + cos x) dx looks hopeless. Read it from the other end (x → π/2 − x) and sine and cosine
swap. Add the two forms and the fraction becomes 1, so twice the integral is π/2, and the integral is **π/4**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top. In the check cell, type a function, its limits and your
answer, then press **Check**: it works out the integral and draws it, slices above the axis in one colour and below
it in another. When a slider appears, drag it. It works in your browser (no install) and in Google Colab. Nothing can
break: if something looks odd, run everything again from the top.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the tools. Run this box first.
import re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, VBox, HBox, Layout
from IPython.display import display
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas and integrals

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})
plt.close(plt.figure())                     # warms up the plotting: the first figure is slow in the browser

x = sp.symbols("x")
NAMES = {"x": x, "e": sp.E, "ln": sp.log, "log": sp.log, "pi": sp.pi, "sqrt": sp.sqrt, "abs": sp.Abs,
         "asin": sp.asin, "atan": sp.atan}
RULES = standard_transformations + (implicit_multiplication_application, convert_xor)

def read(s):
    """Turn what you type into maths: x^2 or x**2, e^x, ln(x), sin(x), sqrt(x), |x - 1| or abs(x - 1), pi."""
    s = re.sub(r"\|([^|]+)\|", r"abs(\1)", s.replace("−", "-").replace("π", "pi"))
    return parse_expr(s, local_dict=NAMES, transformations=RULES)

def say(e):
    """Show an expression the way you'd write it: x^2, |x - 1|, ln(x), pi."""
    s = re.sub(r"Abs\(([^()]*)\)", r"|\1|", str(e))
    return s.replace("**", "^").replace("*", "·").replace("log(", "ln(").replace("pi", "π")

def trap(ys, xs):
    """The trapezium rule: the area under a sampled curve."""
    return float(np.sum((ys[1:] + ys[:-1]) * np.diff(xs)) / 2)

def slices_figure(f, a, b, n=40, title=""):
    """The region between f and the x axis from a to b, as n thin slices: above the axis in lilac, below it in coral."""
    fx = sp.lambdify(x, f, "numpy")
    xs = np.linspace(min(a, b), max(a, b), 400)
    ys = fx(xs) * np.ones_like(xs)
    w = (max(a, b) - min(a, b)) / n
    mids = min(a, b) + (np.arange(n) + 0.5) * w
    hs = fx(mids) * np.ones_like(mids)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.bar(mids, hs, width=w * 0.85, color=[LILAC if h >= 0 else CORAL for h in hs], alpha=0.55)
    ax.plot(xs, ys, color=TEAL, lw=3)
    ax.axhline(0, color=DIM, lw=1.2)
    ax.set_xlabel("x"); ax.set_title(title, fontsize=13)
    display(fig); plt.close(fig)                # shown explicitly, so it also appears inside the check's output box

def definite(f, a, b):
    """The integral (exact if sympy can, else a decimal), and the area (the sizes of the slices added up)."""
    exact = sp.integrate(f, (x, a, b))
    value = float(exact) if exact.is_number else None
    fx = sp.lambdify(x, f, "numpy")
    xs = np.linspace(float(a), float(b), 20001)
    ys = fx(xs) * np.ones_like(xs)
    if value is None: value = trap(ys, xs)
    area = trap(np.abs(ys), xs)
    return exact, value, area

def check(f_s, a_s, b_s, ans_s):
    f, a, b = read(f_s), read(a_s), read(b_s)
    exact, value, area = definite(f, a, b)
    print(f"∫ from {say(a)} to {say(b)} of {say(f)} dx  =  {say(sp.nsimplify(exact)) if exact.is_number else ''}  ≈ {value:.6f}")
    print(f"the area (every slice counted as a size)  ≈ {area:.6f}")
    if ans_s.strip():
        yours = float(read(ans_s))
        print(f"your answer {ans_s} ≈ {yours:.6f}:  " + ("✓ matches the integral" if abs(yours - value) < 1e-6 else "✗ does not match the integral")
              + ("   (but the area is different: some slices are below the axis)" if abs(area - value) > 1e-6 else ""))
    slices_figure(f, float(a), float(b), 40, f"∫ = {value:.4f}    area = {area:.4f}")

def checker(f="", a="", b=""):
    """Four boxes (the function, its limits, your answer) and a Check button."""
    bf = Text(value=f, placeholder="sin(x)", description="function", layout=Layout(width="260px"))
    ba = Text(value=a, placeholder="0", description="from", layout=Layout(width="170px"))
    bb = Text(value=b, placeholder="pi", description="to", layout=Layout(width="190px"))
    bA = Text(value="", placeholder="your answer", description="answer", layout=Layout(width="220px"))
    button = Button(description="Check")
    out = Output()
    def on_check(_):
        out.clear_output()
        with out:
            try: check(bf.value, ba.value, bb.value, bA.value)
            except Exception as err: print("I couldn't read that. Try e.g. sin(x), 0, pi  (", err, ")")
    button.on_click(on_check)
    display(VBox([HBox([bf, ba, bb]), HBox([bA, button]), out]))

## 1 · Thin slices add up to F(b) − F(a)

The integral from 1 to 3 of 2x + 1 adds up thin slices under the line. Drag the slider to use more and thinner
slices. The total settles on **10**, which is F(3) − F(1) with F(x) = x² + x.

In [ ]:
def thin_slices(n=4):
    xs = np.linspace(1, 3, 200)
    w = 2 / n
    mids = 1 + (np.arange(n) + 0.5) * w
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(mids, 2 * mids + 1, width=w * 0.9, color=LILAC, alpha=0.55)
    ax.plot(xs, 2 * xs + 1, color=TEAL, lw=3)
    ax.set_xlim(0, 3.5); ax.set_ylim(0, 8); ax.set_xlabel("x")
    ax.set_title(f"{n} slices add up to {np.sum((2 * mids + 1) * w):.4f}   ·   F(3) − F(1) = 12 − 2 = 10")
    plt.show()

interact(thin_slices, n=IntSlider(value=4, min=1, max=40, step=1, description="slices"));

**Notice.** For a straight line the middle of each slice is exact, so even one slice gives 10. Any F works: x² + x + 7
gives (12 + 7) − (2 + 7), and the 7 cancels.

## 2 · The mirror: replace x by a − x

Reading a curve from the other end flips it in a mirror at the middle, and the area underneath can't change. For
sin x ÷ (sin x + cos x) on [0, π/2], the mirror image is cos x ÷ (cos x + sin x). Move x: at every point the two
heights add to exactly **1**.

In [ ]:
def mirror(at=0.4):
    f = lambda t: np.sin(t) / (np.sin(t) + np.cos(t))
    xs = np.linspace(0, np.pi / 2, 300)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.fill_between(xs, 0, f(xs), color=LILAC, alpha=0.35)
    ax.fill_between(xs, f(xs), 1, color=TEAL, alpha=0.15)
    ax.plot(xs, f(xs), color=TEAL, lw=3, label="sin x ÷ (sin x + cos x)")
    ax.plot(xs, f(np.pi / 2 - xs), color=TEAL, lw=2, ls="--", label="its mirror image")
    ax.plot([at, at], [0, f(at)], color=LILAC, lw=5)
    ax.plot([at, at], [f(at), 1], color=CORAL, lw=5)
    ax.set_ylim(0, 1.15); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title(f"heights at x = {at:.2f}:  {f(at):.3f} + {f(np.pi / 2 - at):.3f} = {f(at) + f(np.pi / 2 - at):.3f}")
    plt.show()

interact(mirror, at=FloatSlider(value=0.4, min=0.0, max=1.57, step=0.05, description="x"));

**Notice.** The two regions together fill a rectangle 1 high and π/2 wide. Each is half of it: **π/4 ≈ 0.785**.

In [ ]:
exact, value, area = definite(sp.sin(x) / (sp.sin(x) + sp.cos(x)), 0, sp.pi / 2)
print(f"∫ from 0 to π/2 of sin x ÷ (sin x + cos x) dx = {value:.6f};   π/4 = {np.pi / 4:.6f}")

## 3 · Even and odd, on [−a, a]

An odd function looks the same after a half turn, so the piece below the axis cancels the piece above. An even one
is its own mirror image, so it's twice the right half.

In [ ]:
for f in [x**3 * sp.cos(x), x**2, x**3 * sp.cos(x) + x**2]:
    print(f"∫ from −2 to 2 of {say(f):<22} = {sp.nsimplify(sp.integrate(f, (x, -2, 2)))}")

## 4 · Check your answer

Type a function, its two limits, and your answer. Use `x^2` for x squared, `pi` for π, `abs(x - 1)` for the modulus
of x − 1. Then press **Check**. It works out the integral, adds up the area, and draws the slices: above the axis in
lilac, below it in coral.

In [ ]:
# Four boxes and a button: type a function, its limits and your answer, then press Check.
checker()

## 5 · Move the top limit

The sine curve from 0 to b. Drag b past π. Past π the curve dips below the axis: its slices count as negative in the
integral, but as sizes in the area. **The integral falls while the area keeps growing.**

In [ ]:
def top_limit(b=2.0):
    xs = np.linspace(0, 2 * np.pi, 400)
    s = np.linspace(0, b, 600)
    I, A = 1 - np.cos(b), trap(np.abs(np.sin(s)), s)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(xs, np.sin(xs), color=TEAL, lw=1.5, alpha=0.5)
    ax.fill_between(s, 0, np.sin(s), where=np.sin(s) >= 0, color=LILAC, alpha=0.5)
    ax.fill_between(s, 0, np.sin(s), where=np.sin(s) < 0, color=CORAL, alpha=0.5)
    ax.plot(s, np.sin(s), color=TEAL, lw=3)
    ax.axhline(0, color=DIM, lw=1.2); ax.axvline(b, color=CHALK, lw=1, ls=":")
    ax.set_xlim(0, 2 * np.pi); ax.set_ylim(-1.3, 1.3); ax.set_xlabel("x")
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi]); ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
    ax.set_title(f"top limit {b:.2f}:   integral = {I:.3f}    area = {A:.3f}")
    plt.show()

interact(top_limit, b=FloatSlider(value=2.0, min=0.0, max=6.25, step=0.05, description="top limit"));

**Notice.** At 3π/2 ≈ 4.71 the integral is 1 but the area is 3: the student's slip, drawn. An integral can be
negative. An area can't.

## The challenge
∫₀⁵ √x ÷ (√x + √(5 − x)) dx. Hint: replace x with 5 − x. Try your answer in the check above: function
`sqrt(x)/(sqrt(x) + sqrt(5 - x))`, from 0 to 5.

## Sources
CBSE Mathematics 041 syllabus 2026–27 (Integrals; Application of Integrals). All problems are our own.
Retrieved 2026-10-24.